In [1]:
# IMPORT & KHỞI TẠO SPARK SESSION

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.fpm import FPGrowth
import datetime

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Khởi tạo Spark với memory đủ cho FP-Growth
spark = (
    SparkSession.builder
    .appName("FPGrowth_Person1")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "8g")          # Tăng nếu bị OOM
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.shuffle.partitions", "200") # Giảm shuffle overhead
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} khởi động thành công.")


Mounted at /content/drive
Spark 4.0.2 khởi động thành công.


In [2]:
# --- Đường dẫn ---
INPUT_FILE       = "/content/drive/MyDrive/FPGROWTH/transactions_train.csv"
DRIVE_BASE       = "/content/drive/MyDrive/FPGROWTH"

PATH_RULES       = f"{DRIVE_BASE}/fp_rules_raw.parquet"
PATH_CANDIDATES  = f"{DRIVE_BASE}/fp_item_to_item_cross_sell.parquet"
PATH_MODEL       = f"{DRIVE_BASE}/fp_model"
PATH_METADATA    = f"{DRIVE_BASE}/fp_metadata.json"


TRAIN_WEEKS = 6
MIN_SUPPORT = 0.0002
MIN_CONFIDENCE = 0.01
MIN_BASKET_SIZE = 2
MAX_BASKET_SIZE = 40
TOP_N_PREDICTIONS = 60


print("Cấu hình đã sẵn sàng!")
print(f"   Input: {INPUT_FILE}")
print(f"   Train: {TRAIN_WEEKS} tuần | minSupport={MIN_SUPPORT} | minConf={MIN_CONFIDENCE}")


Cấu hình đã sẵn sàng!
   Input: /content/drive/MyDrive/FPGROWTH/transactions_train.csv
   Train: 6 tuần | minSupport=0.0002 | minConf=0.01


In [3]:
# ĐỌC DỮ LIỆU & TẠO GIỎ HÀNG
print(f" Đang đọc dữ liệu từ: {INPUT_FILE}")

df = spark.read.option("header", "true").option("inferSchema", "true").csv(INPUT_FILE)

# Xử lý trường hợp CSV không có header
if "_c0" in df.columns:
    df = (df
          .withColumnRenamed("_c0", "t_dat")
          .withColumnRenamed("_c1", "customer_id")
          .withColumnRenamed("_c2", "article_id"))

# Ép kiểu cột ngày
df = df.withColumn("t_dat_date", F.to_date(F.col("t_dat"), "yyyy-MM-dd"))

# Tính khoảng thời gian train/test
max_date         = df.select(F.max("t_dat_date")).collect()[0][0]
test_start_date  = max_date - datetime.timedelta(days=7)
train_start_date = test_start_date - datetime.timedelta(weeks=TRAIN_WEEKS)

print(f"Khoảng train : {train_start_date}  →  {test_start_date}")
print(f"Ground truth : {test_start_date}  →  {max_date}")

# Lọc 6 tuần train
train_raw = df.filter(
    (F.col("t_dat_date") >= train_start_date) &
    (F.col("t_dat_date") <  test_start_date)
)

# Chuẩn hoá article_id: ép 10 chữ số rồi lấy 7 đầu (product_code)
train_raw = train_raw.withColumn(
    "article_id_str",
    F.substring(F.format_string("%010d", F.col("article_id").cast("int")), 1, 7)
)

# Tạo giỏ hàng: gom theo (customer, ngày)
# Dùng collect_set để tự động loại bỏ sản phẩm trùng trong cùng 1 đơn
df_baskets_full = (
    train_raw
    .groupBy("customer_id", "t_dat_date")
    .agg(F.collect_set("article_id_str").alias("items"))
)

# Lọc giỏ hàng hợp lệ
# Bỏ giỏ 1 món và giỏ quá lớn
df_baskets = df_baskets_full.filter(
    (F.size(F.col("items")) > 1) &
    (F.size(F.col("items")) <= MAX_BASKET_SIZE)
)

df_baskets.cache()
total_train_baskets = df_baskets.count()

print(f"\nSchema giỏ hàng:")
df_baskets.printSchema()
print(f"\nSố giỏ hàng hợp lệ đưa vào huấn luyện: {total_train_baskets:,}")
print(f"   minSupport={MIN_SUPPORT} → cặp đồ phải xuất hiện ≥ {int(total_train_baskets * MIN_SUPPORT):,} lần")


 Đang đọc dữ liệu từ: /content/drive/MyDrive/FPGROWTH/transactions_train.csv
Khoảng train : 2020-08-04  →  2020-09-15
Ground truth : 2020-09-15  →  2020-09-22

Schema giỏ hàng:
root
 |-- customer_id: string (nullable = true)
 |-- t_dat_date: date (nullable = true)
 |-- items: array (nullable = false)
 |    |-- element: string (containsNull = false)


Số giỏ hàng hợp lệ đưa vào huấn luyện: 321,308
   minSupport=0.0002 → cặp đồ phải xuất hiện ≥ 64 lần


In [4]:
# HUẤN LUYỆN MÔ HÌNH FP-GROWTH

print(f"Bắt đầu huấn luyện FP-Growth...")

fp = FPGrowth(
    itemsCol="items",
    minSupport=MIN_SUPPORT,
    minConfidence=MIN_CONFIDENCE
)

model = fp.fit(df_baskets)
print("Huấn luyện hoàn tất!")

# Lấy luật thô và kiểm tra sơ bộ
rules_raw = model.associationRules.cache()
total_rules = rules_raw.count()
print(f"   Tổng số luật sinh ra: {total_rules:,}")

# Phân phối lift nhanh
print("\nPhân phối Lift:")
rules_raw.select("lift").summary("min", "25%", "50%", "75%", "max", "mean").show()

good_rules = rules_raw.filter(F.col("lift") > 1.5).count()
print(f"   Luật có lift > 1.5: {good_rules:,} / {total_rules:,}")


Bắt đầu huấn luyện FP-Growth...
Huấn luyện hoàn tất!
   Tổng số luật sinh ra: 1,774

Phân phối Lift:
+-------+------------------+
|summary|              lift|
+-------+------------------+
|    min|0.8113778577116685|
|    25%| 6.463001775274436|
|    50%|13.570105780591543|
|    75%| 32.75079779518422|
|    max|1823.3429726368158|
|   mean| 64.69276315104595|
+-------+------------------+

   Luật có lift > 1.5: 1,744 / 1,774


In [5]:
# TRÍCH XUẤT VÀ MAP LẠI MÃ 10 SỐ CHO BẢN DEMO WEB

print("1. Trích xuất bảng luật trực tiếp từ mô hình FP-Growth (Lift > 1.2)...")
rules = model.associationRules.filter(F.col("lift") > 1.2)

print("1.5. Xây dựng bộ từ điển Mapping (7 số -> 10 số bán chạy nhất)...")
# Khôi phục mã 10 số gốc từ tập train
train_full_id = train_raw.withColumn(
    "article_id_10",
    F.format_string("%010d", F.col("article_id").cast("int"))
)

# Tìm màu sắc (mã 10 số) được mua nhiều nhất cho mỗi kiểu dáng (mã 7 số)
pop_mapping = train_full_id.groupBy("article_id_str", "article_id_10").count()
window_map = Window.partitionBy("article_id_str").orderBy(F.col("count").desc())

best_mapping = pop_mapping.withColumn("rn", F.row_number().over(window_map)) \
    .filter(F.col("rn") == 1) \
    .select(F.col("article_id_str").alias("rec_item_7"), F.col("article_id_10"))

print("2. Phẳng hóa và Map lại mã 10 số chuẩn cho Backend...")
# Phẳng hóa luật và nối mảng giỏ hàng thành Key chuỗi (VD: "0237347,0351484")
cart_to_items = rules.withColumn(
    "rec_item_7", F.explode(F.col("consequent"))
).withColumn(
    "cart_key", F.array_join(F.array_sort(F.col("antecedent")), ",")
)

# NỐI MAPPING: Thay thế mã 7 số bằng mã 10 số
cart_to_items_10 = cart_to_items.join(best_mapping, "rec_item_7", "left")

# Xếp hạng bằng mã 10 số
window_cart = Window.partitionBy("cart_key").orderBy(F.col("lift").desc())

top_cart_recs_df = cart_to_items_10.withColumn(
    "rank", F.row_number().over(window_cart)
).filter(
    F.col("rank") <= 20
).withColumn(
    "rec_struct", F.struct(
        F.col("article_id_10").alias("article_id"), # Trả về đúng 10 số cho Backend
        F.col("rank").cast("int"),
        F.round(F.col("lift"), 4).alias("score"),
        F.lit("fp_growth").alias("method")
    )
).groupBy("cart_key").agg(
    F.collect_list("rec_struct").alias("recommendations")
)

print("3. Chuyển đổi sang Python Dictionary để sẵn sàng lưu JSON...")
cart_recs_dict = {
    row['cart_key']: [rec.asDict() for rec in row['recommendations']]
    for row in top_cart_recs_df.collect()
}
print(f" -> Đã tạo thành công thư viện cho {len(cart_recs_dict):,} trạng thái giỏ hàng.")

1. Trích xuất bảng luật trực tiếp từ mô hình FP-Growth (Lift > 1.2)...
1.5. Xây dựng bộ từ điển Mapping (7 số -> 10 số bán chạy nhất)...
2. Phẳng hóa và Map lại mã 10 số chuẩn cho Backend...
3. Chuyển đổi sang Python Dictionary để sẵn sàng lưu JSON...
 -> Đã tạo thành công thư viện cho 517 trạng thái giỏ hàng.


In [6]:
# TẠO KEY DỰ PHÒNG (DEFAULT_FALLBACK)

print("4. Tính toán danh sách Trending Bestsellers (mã 10 số) cho Fallback...")
recent_start = test_start_date - datetime.timedelta(days=14)

# Dùng train_full_id để lấy đúng mã 10 số có ảnh
bestsellers_df = train_full_id.filter(F.col("t_dat_date") >= recent_start) \
    .groupBy("article_id_10").count() \
    .orderBy(F.col("count").desc()) \
    .limit(20)

fallback_recs = []
for rank, row in enumerate(bestsellers_df.collect(), start=1):
    fallback_recs.append({
        "article_id": row["article_id_10"],
        "rank": rank,
        "score": float(row["count"]),       # Dùng lượt mua thực tế làm điểm tin cậy
        "method": "global_bestseller"
    })

print("5. Gắn Key mặc định 'DEFAULT_FALLBACK' vào từ điển...")
cart_recs_dict["DEFAULT_FALLBACK"] = fallback_recs
print(" -> Đã thêm Fallback thành công.")

4. Tính toán danh sách Trending Bestsellers (mã 10 số) cho Fallback...
5. Gắn Key mặc định 'DEFAULT_FALLBACK' vào từ điển...
 -> Đã thêm Fallback thành công.


In [7]:
# ĐÁNH GIÁ CART-BASED BẰNG KỸ THUẬT "LEAVE-ONE-OUT"

print("Đang đánh giá hiệu năng trên giỏ hàng đang active...")

def evaluate_cart_based(fp_model, full_df, target_start, target_end):
    # 1. Lọc giao dịch trong tuần Test và tạo giỏ hàng thực tế
    test_raw = full_df.filter(
        (F.col("t_dat_date") >= target_start) &
        (F.col("t_dat_date") < target_end)
    ).withColumn(
        "article_id_str",
        F.substring(F.format_string("%010d", F.col("article_id").cast("int")), 1, 7)
    )

    test_baskets = test_raw.groupBy("customer_id", "t_dat_date").agg(
        F.collect_set("article_id_str").alias("full_basket")
    ).filter(F.size(F.col("full_basket")) >= 2)

    # 2. Giả lập giỏ hàng (Leave-One-Out):
    # Lấy món đồ cuối cùng làm Target (món khách sẽ mua thêm)
    # Phần còn lại làm Input (những món đang nằm sẵn trong giỏ)
    simulated_carts = test_baskets.withColumn(
        "cart_input",
        F.expr("slice(full_basket, 1, size(full_basket) - 1)")
    ).withColumn(
        "target_item",
        F.expr("element_at(full_basket, size(full_basket))")
    ).withColumnRenamed("cart_input", "items")

    total_carts = simulated_carts.count()

    # 3. Predict: Dùng model dự đoán các món mua thêm dựa trên 'cart_input'
    predictions = fp_model.transform(simulated_carts)

    # 4. Đếm số giỏ hàng mà FP-Growth đoán trúng món đồ Target bị giấu
    hits_df = predictions.filter(
        (F.size(F.col("prediction")) > 0) &
        F.expr("array_contains(prediction, target_item)")
    )

    hits = hits_df.count()
    recall = hits / total_carts if total_carts > 0 else 0

    print(f"   -> Tổng số giỏ hàng giả lập (>= 2 món) tuần Test: {total_carts:,}")
    print(f"   -> Số giỏ hàng được FP-Growth đoán trúng món tiếp theo (Hits): {hits:,}")
    print(f"   -> Độ bao phủ (Hit Rate): {recall:.4f} ({recall*100:.2f}%)")

evaluate_cart_based(model, df, test_start_date, max_date)

Đang đánh giá hiệu năng trên giỏ hàng đang active...
   -> Tổng số giỏ hàng giả lập (>= 2 món) tuần Test: 45,008
   -> Số giỏ hàng được FP-Growth đoán trúng món tiếp theo (Hits): 5,027
   -> Độ bao phủ (Hit Rate): 0.1117 (11.17%)


In [8]:
# LƯU TẤT CẢ KẾT QUẢ SANG JSON

import json as _json
import os

print("Đang lưu các file output...")

OUTPUT_DIR_SERVING = f"{DRIVE_BASE}/outputs_v2/serving/"
os.makedirs(OUTPUT_DIR_SERVING, exist_ok=True)

PATH_RULES_JSON = f"{DRIVE_BASE}/fp_rules_raw.json"
PATH_CART_RECS_JSON = f"{OUTPUT_DIR_SERVING}cart_recommendations_demo.json"

# 1. Lưu luật thô dưới dạng JSON
print(f"   Lưu rules_raw → {PATH_RULES_JSON}")
rules_raw.write.mode("overwrite").json(PATH_RULES_JSON)

# 2. Lưu từ điển gợi ý mua kèm cho Web Demo
print(f"   Lưu cart_recs_dict → {PATH_CART_RECS_JSON}")
with open(PATH_CART_RECS_JSON, "w", encoding="utf-8") as f:
    _json.dump(cart_recs_dict, f, indent=2)

# 3. Lưu model để tái sử dụng
print(f"   Lưu model → {PATH_MODEL}")
model.write().overwrite().save(PATH_MODEL)

# 4. Lưu metadata
metadata = {
    "test_start_date":         str(test_start_date),
    "train_start_date":        str(train_start_date),
    "max_date":                str(max_date),
    "total_train_baskets":     total_train_baskets,
    "total_rules":             total_rules,
    "total_cart_combinations": len(cart_recs_dict),
    "MIN_SUPPORT":             MIN_SUPPORT,
    "MIN_CONFIDENCE":          MIN_CONFIDENCE,
    "TRAIN_WEEKS":             TRAIN_WEEKS,
    "TOP_N_PREDICTIONS":       TOP_N_PREDICTIONS,
    "INPUT_FILE":              INPUT_FILE,
}
with open(PATH_METADATA, "w") as f:
    _json.dump(metadata, f, indent=2, default=str)
print(f"   Lưu metadata → {PATH_METADATA}")

Đang lưu các file output...
   Lưu rules_raw → /content/drive/MyDrive/FPGROWTH/fp_rules_raw.json
   Lưu cart_recs_dict → /content/drive/MyDrive/FPGROWTH/outputs_v2/serving/cart_recommendations_demo.json
   Lưu model → /content/drive/MyDrive/FPGROWTH/fp_model
   Lưu metadata → /content/drive/MyDrive/FPGROWTH/fp_metadata.json


In [9]:
# Đọc lại file luật thô đã xuất ra ở bước trước
rules_raw = spark.read.json(f"{DRIVE_BASE}/fp_rules_raw.json")

# Sắp xếp theo Lift giảm dần và hiển thị Top 10 luật mạnh nhất
rules_raw.select("antecedent", "consequent", "confidence", "lift") \
         .orderBy(F.col("lift").desc()) \
         .show(10, truncate=False)

# Hoặc sắp xếp theo Confidence để xem những món đồ hay được mua kèm nhất
rules_raw.select("antecedent", "consequent", "confidence", "lift") \
         .orderBy(F.col("confidence").desc()) \
         .show(10, truncate=False)

+----------+----------+------------------+------------------+
|antecedent|consequent|confidence        |lift              |
+----------+----------+------------------+------------------+
|[0909823] |[0909827] |0.7604166666666666|1823.3429726368158|
|[0909827] |[0909823] |0.5447761194029851|1823.3429726368158|
|[0725662] |[0725663] |0.8055555555555556|1479.0368253968256|
|[0725663] |[0725662] |0.6628571428571428|1479.0368253968254|
|[0841565] |[0913540] |0.4827586206896552|1261.0911129800954|
|[0913540] |[0841565] |0.7967479674796748|1261.0911129800954|
|[0917297] |[0917300] |0.5158730158730159|1209.8841385702701|
|[0917300] |[0917297] |0.4744525547445255|1209.88413857027  |
|[0903590] |[0933889] |0.544             |1059.3427393939396|
|[0933889] |[0903590] |0.4121212121212121|1059.3427393939394|
+----------+----------+------------------+------------------+
only showing top 10 rows
+------------------+----------+------------------+------------------+
|antecedent        |consequent|confid